# 🌾 Atelier IA Agricole — 01. SLM: les petits modèles de langage

Un **SLM** (*Small Language Model*) est un modèle de langage **compact** (autour du **milliard
de paramètres**, au lieu de dizaines de milliards pour un LLM).

**Pourquoi est-ce crucial en agriculture?**
- 📶 Fonctionne **hors-ligne** (utile au champ, sans Internet).
- 📱 Tourne sur du **matériel modeste** (téléphone, mini-PC, Raspberry Pi).
- ⚡ **Rapide** et **peu gourmand** en énergie.
- 🔒 Données qui **restent locales** (confidentialité).

Le compromis: un SLM est **moins savant** qu'un gros LLM (notebook 02). Ce notebook n'est pas
un cours de programmation: le code est volontairement court, pour se concentrer sur **comment
utiliser** un modèle de fondation et sur l'effet de réglages comme la **température**, la
**taille de réponse** et le **ingénierie de prompt** (dont l'apprentissage *few-shot*).


In [ ]:
# === Configuration de l'environnement (exécuter en premier) ===
import os, sys, subprocess

# Sommes-nous dans Google Colab?
try:
    import google.colab
    IS_COLAB = True
except Exception:
    IS_COLAB = False

# MODE_DEMO = True  -> utilise de tout petits modèles / jeux de données réduits
# (utile pour tester rapidement, ou hors GPU). Mettez False pour l'atelier complet.
# La variable d'environnement ATELIER_DEMO permet de forcer le mode pour les tests.
MODE_DEMO = os.environ.get("ATELIER_DEMO", "0") == "1"

def pip_install(*paquets):
    """Installe des paquets pip de façon silencieuse (Colab ou local)."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *paquets]
    print("Installation :", " ".join(paquets), "...")
    subprocess.run(cmd, check=False)

print(f"IS_COLAB = {IS_COLAB}")
print(f"MODE_DEMO = {MODE_DEMO}")
print("Python :", sys.version.split()[0])


IS_COLAB = True
MODE_DEMO = False
Python : 3.12.13


In [ ]:
pip_install("transformers>=4.56", "accelerate")
import gc, time
from transformers import pipeline
print("✅ Bibliothèques prêtes.")


Installation : transformers>=4.56 accelerate ...
✅ Bibliothèques prêtes.


## 1. Choisir des SLM à comparer

Deux modèles **ouverts** (aucune clé API requise), du plus petit au plus proche du milliard:

| Modèle (Hugging Face) | Paramètres |
|------------------------|-----------|
| `Qwen/Qwen2.5-0.5B-Instruct`          | 0,5 Md  |
| `TinyLlama/TinyLlama-1.1B-Chat-v1.0`  | 1,1 Md  |

En **mode démo**, on ne charge que le plus petit (pour aller vite). En mode complet, on compare
les deux et on observe l'effet de la taille.


In [ ]:
if MODE_DEMO:
    MODELES_SLM = ["Qwen/Qwen2.5-0.5B-Instruct"]
else:
    MODELES_SLM = ["Qwen/Qwen2.5-0.5B-Instruct", "TinyLlama/TinyLlama-1.1B-Chat-v1.0"]

print("Modèles à comparer:", MODELES_SLM)


Modèles à comparer : ['Qwen/Qwen2.5-0.5B-Instruct', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0']


## 2. Charger et interroger un SLM

On charge chaque modèle avec la fonction `pipeline(...)` de Transformers — la façon la plus
simple d'utiliser un modèle Hugging Face. Un **cache** garde **un seul modèle en mémoire à la
fois** (utile quand la RAM est limitée, comme sur un petit ordinateur ou au champ).


In [ ]:
_pipe_cache = {}

def obtenir_pipeline(nom_modele):
    """Charge un pipeline de génération de texte (et vide le cache précédent pour la RAM)."""
    if nom_modele not in _pipe_cache:
        _pipe_cache.clear()
        gc.collect()
        print(f"Chargement de {nom_modele} ...")
        _pipe_cache[nom_modele] = pipeline("text-generation", model=nom_modele, dtype="auto")
    return _pipe_cache[nom_modele]

def chat(nom_modele, prompt, systeme=None, temperature=0.0):
    """Interroge le SLM.
    - `systeme`        : consigne de rôle (prompt engineering)
    - `temperature`    : créativité (0 = déterministe, 1 = très varié)
    - `max_new_tokens` : taille maximale de la réponse générée
    """
    pipe = obtenir_pipeline(nom_modele)
    messages = ([{"role": "system", "content": systeme}] if systeme else []) + \
               [{"role": "user", "content": prompt}]

    sortie = pipe(messages)
    return sortie[0]["generated_text"][-1]["content"].strip()

# Premier test
print(chat(MODELES_SLM[0], "Qu'est-ce que la rotation des cultures?"))


Chargement de Qwen/Qwen2.5-0.5B-Instruct ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


La rotation des cultures est une théorie économique qui vise à déterminer comment les économies peuvent se développer dans le monde entier en fonction de l'importance et du pouvoir des différents types de cultures. Voici quelques points clés sur cette théorie :

1. Objectif : La rotation des cultures vise à identifier les facteurs d'influence qui contribuent à l'économie mondiale.

2. Types de cultures :
   - Élémentaires (comme les cultures agricoles)
   - Développements (comme les cultures industrielles)
   - Émergences (comme les cultures innovantes)

3. Équilibres et conflits :
   - Les cultures éloignées peuvent s'avérer plus propices à l'économie que les cultures proches.
   - Le développement rapide peut entraîner un érosion des cultures plus anciennes.

4. Processus de rotation :
   - Des processus de rotation sont généralement identifiés pour chaque culture.
   - Ces processus sont souvent associés aux changements sociaux, technologiques ou économiques.

5. Impact sur la socié

## 3. Comparer taille, vitesse et qualité

On pose la **même** question à chaque modèle et on **compare** la réponse.


In [ ]:
question = "Qu'est-ce que la rotation des cultures?"

for m in MODELES_SLM:
    debut = time.time()
    reponse = chat(m, question)
    duree = time.time() - debut
    print(f"\n=== {m} ({duree:.1f} s) ===")
    print(reponse)


Chargement de Qwen/Qwen2.5-0.5B-Instruct ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== Qwen/Qwen2.5-0.5B-Instruct (52.9 s) ===
La rotation des cultures est un concept qui se réfère à l'histoire et aux différences culturelles entre différentes nations ou régions. Cela peut être défini comme le processus de transition d'une culture spécifique vers une autre dans une période d'intervalle, généralement d'un certain nombre de siècles.

Ces changements peuvent venir du développement historique de l'industrie, du commerce, de la technologie, ou même de la politique nationale. Les cultures peuvent aussi évoluer en fonction des traditions, des croyances, des technologies, des modes de vie, etc.

Cette rotation peut avoir des implications significatives pour les relations internationales, les migrations ethniques, et même pour la diversité culturelle sur terre nanteuse. Elle peut également influencer les pratiques sociales, politiques, économiques et religieuses dans chaque culture individuelle.

En termes de philosophie, cette rotation pourrait être considérée comme une form

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


=== TinyLlama/TinyLlama-1.1B-Chat-v1.0 (265.1 s) ===
La rotation des cultures est un processus complexe qui a lieu lorsqu'une culture est migrée de l'un de ses territoires vers un autre. Elle est liée à la migration des populations humaines et à l'essor de l'activité agricole, d'une part, et à la culture d'échanges culturels, d'autre part. 

La rotation des cultures est une évolution phénomène qui s'apparente à la migration, mais avec une durée variable selon les faits. Elle peut avoir lieu chez les populations originelles, lorsque l'environnement reste le même, ou chez les migrants, lorsque le pays d'origine est remplacé par un nouveau. 

La rotation des cultures est un phénomène complexe qui implique des changements dans la structure sociale, économique et culturelle des populations, et des modifications dans la relation entre elles. L'évolution des cultures des populations migrantes est également marquée par la migration, la structure sociale et l'histoire, et elle peut être influe

Ingénierie de prompt

In [ ]:
question = "Tu es un ingénieur agronome francophone. Qu'est-ce que la rotation des cultures?"
debut = time.time()
reponse = chat(m, question)
duree = time.time() - debut
print(f"\n=== {m} ({duree:.1f} s) ===")
print(reponse)


=== TinyLlama/TinyLlama-1.1B-Chat-v1.0 (83.3 s) ===
La rotation des cultures est une pratique agricole utilisée pour rééquilibrer la production agricole. Elle consiste à mélanger les cultures de la même année et de la saison, afin de réduire les risques de sécheresse ou d'emballage de la production. La rotation des cultures est souvent associée à la terroir, c'est-à-dire à la région géographique et à ses habitudes agricoles. Les cultures récupérées ont un plus long cycle de vie à cause de l'intervalle entre leur croissance et la récolte, tandis que les cultures réinvesties ont un cycle de vie plus courte. La rotation des cultures permet de maximiser la productivité de la terre, de réduire les émissions de gaz à effet de serre, de réduire la pollution atmosphérique et de rééquilibrer la production mondiale.


## 4. Un assistant agronome (rôle système)

Le paramètre **`systeme`** donne un **rôle** au modèle, un autre levier d'ingénierie prompt.


In [ ]:
consigne = ("Tu es un ingénieur agronome francophone. "
            "Tu réponds de façon claire, pratique et concise, pour des agriculteurs.")

question = "Qu'est-ce que la rotation des cultures?"
print(chat(MODELES_SLM[0], question, systeme=consigne))


Chargement de Qwen/Qwen2.5-0.5B-Instruct ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


La rotation des cultures est une technique agricole qui consiste à s'adapter à l'agriculture en utilisant différentes variétés d'épices dans les mêmes lieux chaque année. Cela permet de favoriser le développement de diverses variétés et de prévenir les risques de maladies ou de maladies alimentaires. En utilisant la même variété de plantes tous les ans, on peut également réduire les coûts d'entretien et d'amélioration des cultures.


## 5. Un vrai jeu de données agricole (Kaggle)

On utilise le jeu **Crop Recommendation Dataset** (Kaggle): 2200 mesures de sol/climat
(azote N, phosphore P, potassium K, température, humidité, pH, pluviométrie) avec la
**culture recommandée** par des agronomes. `kagglehub` télécharge les jeux **publics** sans
clé API; si Kaggle est injoignable, un petit échantillon de secours prend le relais.


In [ ]:
pip_install("kagglehub", "pandas")
import kagglehub

def telecharger_dataset_kaggle(reference):
    """Télécharge un jeu de données Kaggle public et renvoie son dossier local.
    Renvoie None si Kaggle est injoignable (un échantillon de secours prend alors le relais)."""
    try:
        dossier = kagglehub.dataset_download(reference)
        print(f"✅ Jeu de données Kaggle prêt: {reference}")
        return dossier
    except Exception as e:
        print(f"⚠️ Kaggle indisponible ({e}) → utilisation d'un échantillon de secours.")
        return None


Installation : kagglehub pandas ...


In [ ]:
import pandas as pd

ECHANTILLON_SECOURS = [
    {"N": 20, "P": 129, "K": 201, "temperature": 23.4, "humidity": 91.7, "ph": 5.59, "rainfall": 116.1, "label": "apple"},
    {"N": 100, "P": 80, "K": 52, "temperature": 27.5, "humidity": 77.3, "ph": 6.05, "rainfall": 110.3, "label": "banana"},
    {"N": 43, "P": 68, "K": 20, "temperature": 29.6, "humidity": 66.2, "ph": 7.5, "rainfall": 69.4, "label": "blackgram"},
    {"N": 43, "P": 68, "K": 81, "temperature": 17.5, "humidity": 17.9, "ph": 6.76, "rainfall": 78.9, "label": "chickpea"},
    {"N": 21, "P": 20, "K": 31, "temperature": 25.6, "humidity": 99.7, "ph": 5.86, "rainfall": 165.8, "label": "coconut"},
    {"N": 107, "P": 31, "K": 31, "temperature": 23.2, "humidity": 53.0, "ph": 6.77, "rainfall": 153.1, "label": "coffee"},
    {"N": 122, "P": 40, "K": 17, "temperature": 25.0, "humidity": 81.3, "ph": 6.85, "rainfall": 80.0, "label": "cotton"},
    {"N": 22, "P": 133, "K": 201, "temperature": 23.8, "humidity": 80.1, "ph": 6.0, "rainfall": 67.3, "label": "grapes"},
    {"N": 80, "P": 43, "K": 43, "temperature": 23.8, "humidity": 74.4, "ph": 6.01, "rainfall": 172.6, "label": "jute"},
    {"N": 24, "P": 67, "K": 22, "temperature": 20.1, "humidity": 22.9, "ph": 5.62, "rainfall": 104.6, "label": "kidneybeans"},
    {"N": 18, "P": 66, "K": 22, "temperature": 25.9, "humidity": 67.6, "ph": 6.35, "rainfall": 47.9, "label": "lentil"},
    {"N": 76, "P": 48, "K": 18, "temperature": 19.3, "humidity": 69.6, "ph": 5.78, "rainfall": 83.2, "label": "maize"},
    {"N": 18, "P": 26, "K": 31, "temperature": 32.6, "humidity": 47.7, "ph": 5.42, "rainfall": 91.1, "label": "mango"},
    {"N": 25, "P": 51, "K": 18, "temperature": 27.8, "humidity": 54.8, "ph": 9.46, "rainfall": 50.3, "label": "mothbeans"},
    {"N": 21, "P": 44, "K": 18, "temperature": 27.1, "humidity": 86.9, "ph": 7.13, "rainfall": 50.5, "label": "mungbean"},
    {"N": 100, "P": 17, "K": 48, "temperature": 29.7, "humidity": 94.3, "ph": 6.37, "rainfall": 26.5, "label": "muskmelon"},
    {"N": 12, "P": 20, "K": 10, "temperature": 24.5, "humidity": 93.1, "ph": 6.53, "rainfall": 109.5, "label": "orange"},
    {"N": 54, "P": 67, "K": 52, "temperature": 35.7, "humidity": 93.3, "ph": 6.59, "rainfall": 141.3, "label": "papaya"},
    {"N": 27, "P": 71, "K": 23, "temperature": 23.5, "humidity": 46.5, "ph": 7.11, "rainfall": 150.9, "label": "pigeonpeas"},
    {"N": 21, "P": 21, "K": 38, "temperature": 22.6, "humidity": 89.3, "ph": 6.33, "rainfall": 104.9, "label": "pomegranate"},
    {"N": 81, "P": 53, "K": 42, "temperature": 23.7, "humidity": 81.0, "ph": 5.18, "rainfall": 233.7, "label": "rice"},
    {"N": 103, "P": 16, "K": 49, "temperature": 24.1, "humidity": 81.6, "ph": 6.92, "rainfall": 51.8, "label": "watermelon"},
]

dossier = telecharger_dataset_kaggle("atharvaingle/crop-recommendation-dataset")
df = None
if dossier:
    try:
        df = pd.read_csv(f"{dossier}/Crop_recommendation.csv")
    except Exception as e:
        print(f"⚠️ Lecture du CSV Kaggle impossible ({e}) → échantillon de secours.")
if df is None:
    df = pd.DataFrame(ECHANTILLON_SECOURS)

print(f"{len(df)} lignes disponibles, {df['label'].nunique()} cultures différentes.")
df.head(3)


Using Colab cache for faster access to the 'crop-recommendation-dataset' dataset.
✅ Jeu de données Kaggle prêt: atharvaingle/crop-recommendation-dataset
2200 lignes disponibles, 22 cultures différentes.


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice


## 6. Zero-shot: demander une recommandation sans exemple

On transforme chaque ligne en phrase, puis on demande au SLM de **deviner** la culture sans lui montrer aucun exemple au préalable
# (« zero-shot »).

In [ ]:
def ligne_en_texte(ligne):
    return (f"N={ligne['N']}, P={ligne['P']}, K={ligne['K']}, "
            f"température={ligne['temperature']:.1f}°C, humidité={ligne['humidity']:.0f}%, "
            f"pH={ligne['ph']:.1f}, pluie={ligne['rainfall']:.0f}mm")

LISTE_CULTURES = sorted(df["label"].unique())
CULTURES_TXT = ", ".join(LISTE_CULTURES)

def prompt_zero_shot(ligne):
    return (f"Voici des mesures de sol et de climat: {ligne_en_texte(ligne)}.\n"
            f"Quelle culture recommandes-tu, en choisissant UNIQUEMENT dans cette liste: "
            f"{CULTURES_TXT} ?\nRéponds juste le nom de la culture.")

N_EXEMPLES = 3 if MODE_DEMO else 10
echantillon = df.sample(n=N_EXEMPLES, random_state=42)

for _, ligne in echantillon.iterrows():
    prediction = chat(MODELES_SLM[0], prompt_zero_shot(ligne))
    print(f"Vrai: {ligne['label']:12s} | Prédit (zero-shot): {prediction}")


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Vrai: muskmelon    | Prédit (zero-shot): Blackgram


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Vrai: watermelon   | Prédit (zero-shot): Blackgram


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Vrai: apple        | Prédit (zero-shot): coconut


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Vrai: mothbeans    | Prédit (zero-shot): Apple


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Vrai: mungbean     | Prédit (zero-shot): apple
Vrai: lentil       | Prédit (zero-shot): Blackgram


## 7. Few-shot: donner 3 ou plus d'exemples avant de demander

# Le **prompt engineering** le plus efficace pour guider un petit modèle: montrer quelques
exemples résolus (« few-shot ») avant la vraie question. On réutilise le **même** échantillon
pour comparer équitablement avec le zero-shot.


In [ ]:
exemples_fewshot = df.drop(echantillon.index).sample(n=3, random_state=7)
ex_prompt = ""
for _, ligne in exemples_fewshot.iterrows():
    ex_prompt += prompt_zero_shot(ligne) + f"Réponse: {ligne['label']:12s}"
print(ex_prompt)

print("--- Test Few-Shot ---")
for _, ligne in echantillon.iterrows():
    prompt_few_shot = ex_prompt + prompt_zero_shot(ligne)
    prediction = chat(MODELES_SLM[0], prompt_few_shot)
    print(f"Vrai: {ligne['label']:12s} | Prédit (few-shot): {prediction}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Voici des mesures de sol et de climat: N=11, P=132, K=197, température=16.0°C, humidité=81%, pH=5.7, pluie=74mm.
Quelle culture recommandes-tu, en choisissant UNIQUEMENT dans cette liste: apple, banana, blackgram, chickpea, coconut, coffee, cotton, grapes, jute, kidneybeans, lentil, maize, mango, mothbeans, mungbean, muskmelon, orange, papaya, pigeonpeas, pomegranate, rice, watermelon ?
Réponds uniquement par le nom de la culture.Réponse: grapes      Voici des mesures de sol et de climat: N=13, P=8, K=12, température=25.2°C, humidité=93%, pH=7.1, pluie=114mm.
Quelle culture recommandes-tu, en choisissant UNIQUEMENT dans cette liste: apple, banana, blackgram, chickpea, coconut, coffee, cotton, grapes, jute, kidneybeans, lentil, maize, mango, mothbeans, mungbean, muskmelon, orange, papaya, pigeonpeas, pomegranate, rice, watermelon ?
Réponds uniquement par le nom de la culture.Réponse: orange      Voici des mesures de sol et de climat: N=94, P=53, K=40, température=20.3°C, humidité=83%, p

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Vrai: muskmelon    | Prédit (few-shot): grapes


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Vrai: apple        | Prédit (few-shot): grapefruit


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Vrai: mothbeans    | Prédit (few-shot): grapefruit


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Vrai: mungbean     | Prédit (few-shot): coconut
Vrai: lentil       | Prédit (few-shot): grapefruit


## ✅ Récapitulatif

- Un **SLM** échange un peu de « savoir » contre **rapidité, taille réduite et fonctionnement
  hors-ligne**.
- La **température** contrôle la créativité, `max_new_tokens` la longueur de la réponse.
- Le **few-shot prompting** (montrer des exemples) guide un petit modèle sans le réentraîner.

**➡️ Notebook suivant: `02_LLM_quantization.ipynb`** pour améliorer les réponses.
